In [ ]:
#@title Cell 25.1 - Notebook overview
from IPython.display import display, Markdown

display(Markdown(r"""
# Notebook 25: Model C Pathogen-Kernel Candidate Construction

## Purpose

Notebook 25 will combine the existing Model 3B pathogen kernel with the two
targeted nucleotide-sequence kernels produced in Notebook 24.

It will:

1. load and validate the Notebook 24 sequence-kernel archive;
2. load the existing 9,377-pathogen Model 3B kernel;
3. retain the 9,058 Model C pathogens with assembly sequences;
4. confirm that the pathogen order is identical in every input;
5. define the Model C combination rule

\[
K_P^{(C)}(\rho)
=
\rho K_{\mathrm{seq}}
+(1-\rho)K_P^{(3B)},
\qquad \rho\in[0,1];
\]

6. apply this rule separately to the base-weighted and locus-weighted sequence
   kernels;
7. define the two Model C pathogen-kernel families

\[
K_{P,C,\mathrm{base}}(\rho),
\qquad
K_{P,C,\mathrm{locus}}(\rho)
\in\mathbb{R}^{9058\times9058};
\]

8. validate the component kernels and the candidate values of \(\rho\);
9. save the aligned component matrices and their fixed pathogen order; and
10. package the outputs required for later BioSample-grouped pathogen-out
    validation.

## Model boundary

The existing Model 3B kernel and its 9,377-pathogen results will not be changed.
Notebook 25 will extract the required 9,058-pathogen subset without overwriting
the original files.

The 319 pathogens without assembly sequences will not be reintroduced into
Model C.

The same candidate values of \(\rho\) will be used for the base-weighted and
locus-weighted sequence kernels. Notebook 25 will not select a sequence-kernel
type or a value of \(\rho\).

The antibiotic kernel will remain unchanged. MIC outcomes will not be used in
this notebook.

Spectral conversion, pathogen--antibiotic interaction features, MIC-model
fitting and BioSample-grouped pathogen-out selection are outside the scope of
this notebook.

## Expected notebook length

Notebook 25 contains **10 cells**.
"""))

print(
    "Transition: The next cell will import the required packages "
    "and define the Notebook 25 settings."
)


In [ ]:
# =============================================================================
# Cell 25.2
# =============================================================================

#@title Cell 25.2 - Import packages and define notebook settings
# This cell imports the required packages, mounts Google Drive, defines the
# expected dimensions and specifies the candidate rho values that will later
# be evaluated using BioSample-grouped pathogen-out validation.

import hashlib
import json
import os
import shutil
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd

from google.colab import drive
from IPython.display import display


drive.mount(
    "/content/drive",
    force_remount=False,
)


EXPECTED_MODEL3B_PATHOGENS = 9377
EXPECTED_MODEL_C_PATHOGENS = 9058

RHO_CANDIDATES = np.round(
    np.linspace(
        0.0,
        1.0,
        11,
    ),
    2,
)

ROW_BLOCK_SIZE = 256
VALIDATION_SAMPLE_SIZE = 512

MYDRIVE_DIRECTORY = Path(
    "/content/drive/MyDrive"
)

PROJECT_DIRECTORY = (
    MYDRIVE_DIRECTORY
    / "Model3_MIC_Project"
)

NOTEBOOK24_DIRECTORY = (
    PROJECT_DIRECTORY
    / "notebook24"
)

NOTEBOOK25_DIRECTORY = (
    PROJECT_DIRECTORY
    / "notebook25"
)

NOTEBOOK25_RESULT_DIRECTORY = (
    NOTEBOOK25_DIRECTORY
    / "results"
)

WORK_DIRECTORY = Path(
    "/content/notebook25_work"
)

INPUT_DIRECTORY = (
    WORK_DIRECTORY
    / "inputs"
)

MODEL3B_INPUT_DIRECTORY = (
    INPUT_DIRECTORY
    / "model3b"
)

NOTEBOOK24_INPUT_DIRECTORY = (
    INPUT_DIRECTORY
    / "notebook24"
)

for directory in [
    PROJECT_DIRECTORY,
    NOTEBOOK25_DIRECTORY,
    NOTEBOOK25_RESULT_DIRECTORY,
    WORK_DIRECTORY,
    INPUT_DIRECTORY,
    MODEL3B_INPUT_DIRECTORY,
    NOTEBOOK24_INPUT_DIRECTORY,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


MODEL3B_ARCHIVE_FILENAME = (
    "15B_full_gene_genomic_antibiotic_kernels_outputs.zip"
)

NOTEBOOK24_ARCHIVE_FILENAME = (
    "24_targeted_amr_sequence_kernel_candidates_outputs.zip"
)


def file_sha256(file_path):
    digest = hashlib.sha256()

    with open(
        file_path,
        "rb",
    ) as input_file:
        for block in iter(
            lambda: input_file.read(1024 * 1024),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()


settings_summary = pd.DataFrame(
    [
        {
            "setting": "Model 3B pathogens",
            "value": EXPECTED_MODEL3B_PATHOGENS,
        },
        {
            "setting": "Model C pathogens",
            "value": EXPECTED_MODEL_C_PATHOGENS,
        },
        {
            "setting": "Sequence-kernel candidates",
            "value": 2,
        },
        {
            "setting": "Candidate rho values",
            "value": len(RHO_CANDIDATES),
        },
        {
            "setting": "Rho range",
            "value": "0.0 to 1.0 in steps of 0.1",
        },
        {
            "setting": "Notebook 25 output directory",
            "value": str(NOTEBOOK25_DIRECTORY),
        },
    ]
)

display(settings_summary)

print(
    "Notebook 25 packages, directories and fixed settings "
    "were defined successfully."
)

print(
    "\nTransition: Cell 25.3 will locate and validate the "
    "Notebook 15B and Notebook 24 input archives."
)


In [ ]:
# =============================================================================
# Cell 25.3
# =============================================================================

#@title Cell 25.3 - Locate, validate and extract the required input files
# This cell locates the original Notebook 15B kernel archive and the Notebook
# 24 sequence-kernel archive, validates their member lists and extracts only
# the files required by Notebook 25.


def locate_unique_archive(
    archive_filename,
    preferred_paths,
):
    candidates = []

    for candidate_path in preferred_paths:
        candidate_path = Path(candidate_path)

        if candidate_path.exists():
            candidates.append(
                candidate_path.resolve()
            )

    if not candidates:
        candidates = [
            path.resolve()
            for path in PROJECT_DIRECTORY.rglob(
                archive_filename
            )
        ]

    candidates = sorted(
        set(candidates),
        key=lambda path: (
            len(path.parts),
            str(path),
        ),
    )

    if not candidates:
        raise FileNotFoundError(
            f"{archive_filename} was not found. Copy this "
            f"archive into {PROJECT_DIRECTORY} and rerun "
            "Cell 25.3."
        )

    if len(candidates) > 1:
        candidate_hashes = {
            file_sha256(path)
            for path in candidates
        }

        if len(candidate_hashes) > 1:
            raise ValueError(
                f"Multiple different copies of "
                f"{archive_filename} were found: "
                f"{candidates}. Retain one authoritative "
                "copy and rerun Cell 25.3."
            )

    return candidates[0]


def archive_member_for_basename(
    archive,
    required_basename,
):
    matches = [
        member
        for member in archive.namelist()
        if Path(member).name
        == required_basename
    ]

    if len(matches) != 1:
        raise ValueError(
            f"Expected exactly one archive member named "
            f"{required_basename}, but found {matches}."
        )

    return matches[0]


def validate_and_extract_members(
    archive_path,
    required_basenames,
    destination_directory,
):
    extracted_paths = {}

    with zipfile.ZipFile(
        archive_path,
        "r",
    ) as archive:
        damaged_member = archive.testzip()

        if damaged_member is not None:
            raise ValueError(
                f"{archive_path.name} contains a damaged "
                f"member: {damaged_member}"
            )

        for required_basename in required_basenames:
            archive_member = archive_member_for_basename(
                archive,
                required_basename,
            )

            output_path = (
                destination_directory
                / required_basename
            )

            partial_path = output_path.with_suffix(
                output_path.suffix + ".partial"
            )

            partial_path.unlink(
                missing_ok=True
            )

            with archive.open(
                archive_member,
                "r",
            ) as source_file:
                with open(
                    partial_path,
                    "wb",
                ) as destination_file:
                    shutil.copyfileobj(
                        source_file,
                        destination_file,
                        length=1024 * 1024,
                    )

            partial_path.replace(
                output_path
            )

            extracted_paths[
                required_basename
            ] = output_path

    return extracted_paths


model3b_archive_path = locate_unique_archive(
    MODEL3B_ARCHIVE_FILENAME,
    [
        PROJECT_DIRECTORY
        / MODEL3B_ARCHIVE_FILENAME,
        PROJECT_DIRECTORY
        / "notebook15B"
        / MODEL3B_ARCHIVE_FILENAME,
        MYDRIVE_DIRECTORY
        / MODEL3B_ARCHIVE_FILENAME,
    ],
)

notebook24_archive_path = locate_unique_archive(
    NOTEBOOK24_ARCHIVE_FILENAME,
    [
        NOTEBOOK24_DIRECTORY
        / NOTEBOOK24_ARCHIVE_FILENAME,
        PROJECT_DIRECTORY
        / NOTEBOOK24_ARCHIVE_FILENAME,
    ],
)


model3b_required_files = [
    "15B_pathogen_combined_reference_kernel.npz",
    "15B_pathogen_kernel_index.csv",
    "15B_kernel_configuration.csv",
]

notebook24_required_files = [
    "24_sequence_kernel.npy",
    "24_sequence_kernel_locus_weighted.npy",
    "24_model_c_pathogen_order.csv",
    "24_sequence_kernel_metadata.json",
    "24_sequence_kernel_locus_weighted_metadata.json",
    "24_sequence_kernel_candidate_validation.csv",
    "24_output_manifest.json",
]


model3b_input_paths = validate_and_extract_members(
    model3b_archive_path,
    model3b_required_files,
    MODEL3B_INPUT_DIRECTORY,
)

notebook24_input_paths = validate_and_extract_members(
    notebook24_archive_path,
    notebook24_required_files,
    NOTEBOOK24_INPUT_DIRECTORY,
)


input_archive_summary = pd.DataFrame(
    [
        {
            "input": "Model 3B pathogen kernel",
            "archive": model3b_archive_path.name,
            "required_files": len(
                model3b_required_files
            ),
            "validation_status": "passed",
        },
        {
            "input": "Notebook 24 sequence kernels",
            "archive": notebook24_archive_path.name,
            "required_files": len(
                notebook24_required_files
            ),
            "validation_status": "passed",
        },
    ]
)

display(input_archive_summary)

print(f"Model 3B archive: {model3b_archive_path}")
print(f"Notebook 24 archive: {notebook24_archive_path}")

print(
    "\nTransition: Cell 25.4 will load the Model 3B "
    "kernel, both sequence kernels and their pathogen indexes."
)


In [ ]:
# =============================================================================
# Cell 25.4
# =============================================================================

#@title Cell 25.4 - Load and validate the kernel inputs
# This cell loads the complete Model 3B pathogen kernel, the two Notebook 24
# sequence kernels and the index tables that define their pathogen order.

model3b_kernel_archive = np.load(
    model3b_input_paths[
        "15B_pathogen_combined_reference_kernel.npz"
    ],
    allow_pickle=False,
)

if "combined_reference" not in model3b_kernel_archive.files:
    raise KeyError(
        "The Model 3B kernel archive does not contain "
        "the combined_reference matrix."
    )

model3b_kernel_full = np.asarray(
    model3b_kernel_archive[
        "combined_reference"
    ],
    dtype=np.float32,
)

model3b_pathogen_index = pd.read_csv(
    model3b_input_paths[
        "15B_pathogen_kernel_index.csv"
    ]
)

model3b_configuration = pd.read_csv(
    model3b_input_paths[
        "15B_kernel_configuration.csv"
    ]
)

base_sequence_kernel = np.load(
    notebook24_input_paths[
        "24_sequence_kernel.npy"
    ],
    mmap_mode="r",
)

locus_sequence_kernel = np.load(
    notebook24_input_paths[
        "24_sequence_kernel_locus_weighted.npy"
    ],
    mmap_mode="r",
)

model_c_pathogen_order = pd.read_csv(
    notebook24_input_paths[
        "24_model_c_pathogen_order.csv"
    ]
)

notebook24_kernel_validation = pd.read_csv(
    notebook24_input_paths[
        "24_sequence_kernel_candidate_validation.csv"
    ]
)


if model3b_kernel_full.shape != (
    EXPECTED_MODEL3B_PATHOGENS,
    EXPECTED_MODEL3B_PATHOGENS,
):
    raise ValueError(
        "The Model 3B pathogen kernel has the wrong "
        f"dimensions: {model3b_kernel_full.shape}."
    )

expected_model_c_shape = (
    EXPECTED_MODEL_C_PATHOGENS,
    EXPECTED_MODEL_C_PATHOGENS,
)

for kernel_name, kernel in [
    (
        "Base-weighted sequence kernel",
        base_sequence_kernel,
    ),
    (
        "Locus-weighted sequence kernel",
        locus_sequence_kernel,
    ),
]:
    if kernel.shape != expected_model_c_shape:
        raise ValueError(
            f"{kernel_name} has dimensions "
            f"{kernel.shape}; expected "
            f"{expected_model_c_shape}."
        )

required_model3b_index_columns = {
    "kernel_row",
    "biosample",
}

required_model_c_order_columns = {
    "model_c_row_index",
    "model_3b_row_index",
    "biosample",
    "assembly_accession",
}

if not required_model3b_index_columns.issubset(
    model3b_pathogen_index.columns
):
    raise ValueError(
        "The Model 3B pathogen index is missing required "
        "columns."
    )

if not required_model_c_order_columns.issubset(
    model_c_pathogen_order.columns
):
    raise ValueError(
        "The Model C pathogen order is missing required "
        "columns."
    )

if len(model3b_pathogen_index) != (
    EXPECTED_MODEL3B_PATHOGENS
):
    raise ValueError(
        "The Model 3B pathogen index has the wrong "
        "number of rows."
    )

if len(model_c_pathogen_order) != (
    EXPECTED_MODEL_C_PATHOGENS
):
    raise ValueError(
        "The Model C pathogen order has the wrong "
        "number of rows."
    )

if not (
    notebook24_kernel_validation[
        "validation_status"
    ]
    == "passed"
).all():
    raise ValueError(
        "At least one Notebook 24 sequence kernel did "
        "not pass its recorded validation."
    )


loaded_input_summary = pd.DataFrame(
    [
        {
            "input": "Complete Model 3B pathogen kernel",
            "dimensions": "9,377 × 9,377",
            "status": "loaded",
        },
        {
            "input": "Base-weighted sequence kernel",
            "dimensions": "9,058 × 9,058",
            "status": "loaded",
        },
        {
            "input": "Locus-weighted sequence kernel",
            "dimensions": "9,058 × 9,058",
            "status": "loaded",
        },
    ]
)

display(loaded_input_summary)

print(
    "\nTransition: Cell 25.5 will align the pathogen "
    "indexes and extract the 9,058-pathogen Model 3B subset."
)


In [ ]:
# =============================================================================
# Cell 25.5
# =============================================================================

#@title Cell 25.5 - Align pathogen order and extract the Model 3B subset
# This cell confirms that every Notebook 24 pathogen refers to the expected
# Model 3B kernel row, extracts those rows and columns in the same order as the
# sequence kernels, and saves the resulting 9,058 by 9,058 matrix.

model_c_pathogen_order = (
    model_c_pathogen_order
    .sort_values(
        "model_c_row_index"
    )
    .reset_index(drop=True)
)

expected_model_c_rows = np.arange(
    EXPECTED_MODEL_C_PATHOGENS,
    dtype=np.int64,
)

observed_model_c_rows = model_c_pathogen_order[
    "model_c_row_index"
].to_numpy(dtype=np.int64)

if not np.array_equal(
    observed_model_c_rows,
    expected_model_c_rows,
):
    raise ValueError(
        "The Model C row index is not the complete ordered "
        "range from 0 to 9,057."
    )

model3b_rows = model_c_pathogen_order[
    "model_3b_row_index"
].to_numpy(dtype=np.int64)

if (
    model3b_rows.min() < 0
    or model3b_rows.max()
    >= EXPECTED_MODEL3B_PATHOGENS
):
    raise ValueError(
        "At least one Model 3B row index is outside the "
        "9,377-pathogen kernel."
    )

if len(np.unique(model3b_rows)) != (
    EXPECTED_MODEL_C_PATHOGENS
):
    raise ValueError(
        "The Model C manifest contains duplicate Model 3B "
        "row indexes."
    )

model3b_index_by_row = (
    model3b_pathogen_index
    .assign(
        kernel_row=lambda table: table[
            "kernel_row"
        ].astype(np.int64),
        biosample=lambda table: table[
            "biosample"
        ].astype(str),
    )
    .set_index("kernel_row")
)

model3b_ordered_biosamples = (
    model3b_index_by_row
    .loc[model3b_rows, "biosample"]
    .to_numpy(dtype=str)
)

model_c_ordered_biosamples = (
    model_c_pathogen_order[
        "biosample"
    ]
    .astype(str)
    .to_numpy()
)

if not np.array_equal(
    model3b_ordered_biosamples,
    model_c_ordered_biosamples,
):
    mismatch_positions = np.flatnonzero(
        model3b_ordered_biosamples
        != model_c_ordered_biosamples
    )

    first_mismatch = int(
        mismatch_positions[0]
    )

    raise ValueError(
        "Model 3B and Notebook 24 pathogen order differ at "
        f"Model C row {first_mismatch}: "
        f"{model3b_ordered_biosamples[first_mismatch]} "
        f"versus "
        f"{model_c_ordered_biosamples[first_mismatch]}."
    )


MODEL3B_SUBSET_KERNEL_PATH = (
    NOTEBOOK25_RESULT_DIRECTORY
    / "25_model3b_pathogen_kernel_subset.npy"
)

MODEL_C_PATHOGEN_ORDER_PATH = (
    NOTEBOOK25_RESULT_DIRECTORY
    / "25_model_c_pathogen_order.csv"
)

local_subset_kernel_path = (
    WORK_DIRECTORY
    / MODEL3B_SUBSET_KERNEL_PATH.name
)

local_subset_kernel_path.unlink(
    missing_ok=True
)

model3b_subset_writer = np.lib.format.open_memmap(
    local_subset_kernel_path,
    mode="w+",
    dtype=np.float32,
    shape=(
        EXPECTED_MODEL_C_PATHOGENS,
        EXPECTED_MODEL_C_PATHOGENS,
    ),
)

for row_start in range(
    0,
    EXPECTED_MODEL_C_PATHOGENS,
    ROW_BLOCK_SIZE,
):
    row_stop = min(
        row_start + ROW_BLOCK_SIZE,
        EXPECTED_MODEL_C_PATHOGENS,
    )

    model3b_subset_writer[
        row_start:row_stop,
        :,
    ] = model3b_kernel_full[
        np.ix_(
            model3b_rows[row_start:row_stop],
            model3b_rows,
        )
    ]

model3b_subset_writer.flush()
del model3b_subset_writer

partial_subset_path = (
    MODEL3B_SUBSET_KERNEL_PATH.with_suffix(
        ".npy.partial"
    )
)

partial_subset_path.unlink(
    missing_ok=True
)

shutil.copy2(
    local_subset_kernel_path,
    partial_subset_path,
)

if (
    partial_subset_path.stat().st_size
    != local_subset_kernel_path.stat().st_size
):
    raise IOError(
        "The copied Model 3B subset has the wrong file size."
    )

partial_subset_path.replace(
    MODEL3B_SUBSET_KERNEL_PATH
)

model_c_pathogen_order.to_csv(
    MODEL_C_PATHOGEN_ORDER_PATH,
    index=False,
)

model3b_subset_kernel = np.load(
    MODEL3B_SUBSET_KERNEL_PATH,
    mmap_mode="r",
)

del model3b_kernel_full
model3b_kernel_archive.close()


order_summary = pd.DataFrame(
    [
        {
            "metric": "Model C pathogens",
            "value": EXPECTED_MODEL_C_PATHOGENS,
        },
        {
            "metric": "Unique Model 3B rows retained",
            "value": len(
                np.unique(model3b_rows)
            ),
        },
        {
            "metric": "BioSample order matches",
            "value": True,
        },
        {
            "metric": "Model 3B subset dimensions",
            "value": "9,058 × 9,058",
        },
    ]
)

display(order_summary)

print(f"Saved: {MODEL3B_SUBSET_KERNEL_PATH}")
print(f"Saved: {MODEL_C_PATHOGEN_ORDER_PATH}")

print(
    "\nTransition: Cell 25.6 will validate the aligned "
    "Model 3B subset and both sequence-kernel components."
)


In [ ]:
# =============================================================================
# Cell 25.6
# =============================================================================

#@title Cell 25.6 - Validate the three aligned kernel components
# This cell validates the Model 3B subset and both sequence kernels using the
# same pathogen order. It confirms dimensions, finite values, symmetry,
# diagonal, numerical range and sampled positive-semidefinite behaviour.


def validate_kernel_component(
    kernel,
    kernel_name,
):
    expected_shape = (
        EXPECTED_MODEL_C_PATHOGENS,
        EXPECTED_MODEL_C_PATHOGENS,
    )

    if kernel.shape != expected_shape:
        raise ValueError(
            f"{kernel_name} has dimensions {kernel.shape}; "
            f"expected {expected_shape}."
        )

    kernel_minimum = np.inf
    kernel_maximum = -np.inf
    all_finite = True
    maximum_symmetry_difference = 0.0

    for row_start in range(
        0,
        EXPECTED_MODEL_C_PATHOGENS,
        ROW_BLOCK_SIZE,
    ):
        row_stop = min(
            row_start + ROW_BLOCK_SIZE,
            EXPECTED_MODEL_C_PATHOGENS,
        )

        row_block = np.asarray(
            kernel[row_start:row_stop, :]
        )

        column_block = np.asarray(
            kernel[:, row_start:row_stop]
        ).T

        all_finite = (
            all_finite
            and np.isfinite(row_block).all()
        )

        kernel_minimum = min(
            kernel_minimum,
            float(row_block.min()),
        )

        kernel_maximum = max(
            kernel_maximum,
            float(row_block.max()),
        )

        maximum_symmetry_difference = max(
            maximum_symmetry_difference,
            float(
                np.max(
                    np.abs(
                        row_block
                        - column_block
                    )
                )
            ),
        )

    diagonal = np.asarray(
        np.diagonal(kernel),
        dtype=np.float64,
    )

    maximum_diagonal_difference = float(
        np.max(
            np.abs(
                diagonal - 1.0
            )
        )
    )

    sample_size = min(
        VALIDATION_SAMPLE_SIZE,
        EXPECTED_MODEL_C_PATHOGENS,
    )

    sample_indices = np.linspace(
        0,
        EXPECTED_MODEL_C_PATHOGENS - 1,
        sample_size,
        dtype=int,
    )

    sample_kernel = np.asarray(
        kernel[
            np.ix_(
                sample_indices,
                sample_indices,
            )
        ],
        dtype=np.float64,
    )

    sample_minimum_eigenvalue = float(
        np.linalg.eigvalsh(
            sample_kernel
        ).min()
    )

    validation_checks = {
        "all_values_finite": all_finite,
        "symmetric": (
            maximum_symmetry_difference
            <= 1e-6
        ),
        "diagonal_correct": (
            maximum_diagonal_difference
            <= 1e-6
        ),
        "values_in_zero_one_range": (
            kernel_minimum >= -1e-6
            and kernel_maximum <= 1.0 + 1e-6
        ),
        "sample_principal_matrix_psd": (
            sample_minimum_eigenvalue
            >= -1e-4
        ),
    }

    if not all(validation_checks.values()):
        failed_checks = [
            check_name
            for check_name, passed
            in validation_checks.items()
            if not passed
        ]

        raise ValueError(
            f"{kernel_name} validation failed: "
            f"{failed_checks}"
        )

    return {
        "kernel_component": kernel_name,
        "rows": kernel.shape[0],
        "columns": kernel.shape[1],
        "minimum_value": kernel_minimum,
        "maximum_value": kernel_maximum,
        "maximum_symmetry_difference":
            maximum_symmetry_difference,
        "maximum_diagonal_difference":
            maximum_diagonal_difference,
        "sample_minimum_eigenvalue":
            sample_minimum_eigenvalue,
        "validation_status": "passed",
    }


component_validation = pd.DataFrame(
    [
        validate_kernel_component(
            model3b_subset_kernel,
            "Model 3B pathogen-kernel subset",
        ),
        validate_kernel_component(
            base_sequence_kernel,
            "Base-weighted sequence kernel",
        ),
        validate_kernel_component(
            locus_sequence_kernel,
            "Locus-weighted sequence kernel",
        ),
    ]
)

KERNEL_COMPONENT_VALIDATION_PATH = (
    NOTEBOOK25_RESULT_DIRECTORY
    / "25_kernel_component_validation.csv"
)

component_validation.to_csv(
    KERNEL_COMPONENT_VALIDATION_PATH,
    index=False,
)

display(component_validation)

print(f"Saved: {KERNEL_COMPONENT_VALIDATION_PATH}")

print(
    "\nTransition: Cell 25.7 will define and validate "
    "both Model C kernel families across the candidate rho values."
)


In [ ]:
# =============================================================================
# Cell 25.7
# =============================================================================

#@title Cell 25.7 - Define and validate both Model C kernel families
# This cell applies the agreed convex-combination formula to sampled principal
# matrices for every candidate rho value. It validates both sequence-kernel
# families without selecting rho or using MIC outcomes.

RHO_CANDIDATE_PATH = (
    NOTEBOOK25_RESULT_DIRECTORY
    / "25_rho_candidate_grid.csv"
)

KERNEL_FAMILY_VALIDATION_PATH = (
    NOTEBOOK25_RESULT_DIRECTORY
    / "25_kernel_family_sample_validation.csv"
)

rho_candidate_table = pd.DataFrame(
    {
        "rho": RHO_CANDIDATES,
        "model3b_contribution": (
            1.0 - RHO_CANDIDATES
        ),
        "sequence_contribution":
            RHO_CANDIDATES,
    }
)

rho_candidate_table.to_csv(
    RHO_CANDIDATE_PATH,
    index=False,
)

sample_size = min(
    VALIDATION_SAMPLE_SIZE,
    EXPECTED_MODEL_C_PATHOGENS,
)

sample_indices = np.linspace(
    0,
    EXPECTED_MODEL_C_PATHOGENS - 1,
    sample_size,
    dtype=int,
)

model3b_sample = np.asarray(
    model3b_subset_kernel[
        np.ix_(
            sample_indices,
            sample_indices,
        )
    ],
    dtype=np.float64,
)

sequence_samples = {
    "base-weighted": np.asarray(
        base_sequence_kernel[
            np.ix_(
                sample_indices,
                sample_indices,
            )
        ],
        dtype=np.float64,
    ),
    "locus-weighted": np.asarray(
        locus_sequence_kernel[
            np.ix_(
                sample_indices,
                sample_indices,
            )
        ],
        dtype=np.float64,
    ),
}

kernel_family_validation_rows = []

for sequence_kernel_name, sequence_sample in (
    sequence_samples.items()
):
    for rho in RHO_CANDIDATES:
        combined_sample = (
            (1.0 - rho)
            * model3b_sample
            + rho
            * sequence_sample
        )

        maximum_symmetry_difference = float(
            np.max(
                np.abs(
                    combined_sample
                    - combined_sample.T
                )
            )
        )

        maximum_diagonal_difference = float(
            np.max(
                np.abs(
                    np.diag(combined_sample)
                    - 1.0
                )
            )
        )

        minimum_eigenvalue = float(
            np.linalg.eigvalsh(
                combined_sample
            ).min()
        )

        validation_passed = (
            np.isfinite(combined_sample).all()
            and float(combined_sample.min())
            >= -1e-6
            and float(combined_sample.max())
            <= 1.0 + 1e-6
            and maximum_symmetry_difference
            <= 1e-6
            and maximum_diagonal_difference
            <= 1e-6
            and minimum_eigenvalue
            >= -1e-4
        )

        kernel_family_validation_rows.append(
            {
                "sequence_kernel":
                    sequence_kernel_name,
                "rho": float(rho),
                "model3b_contribution": float(
                    1.0 - rho
                ),
                "sequence_contribution": float(rho),
                "sample_minimum_value": float(
                    combined_sample.min()
                ),
                "sample_maximum_value": float(
                    combined_sample.max()
                ),
                "maximum_symmetry_difference":
                    maximum_symmetry_difference,
                "maximum_diagonal_difference":
                    maximum_diagonal_difference,
                "sample_minimum_eigenvalue":
                    minimum_eigenvalue,
                "validation_status": (
                    "passed"
                    if validation_passed
                    else "failed"
                ),
            }
        )

kernel_family_validation = pd.DataFrame(
    kernel_family_validation_rows
)

if not (
    kernel_family_validation[
        "validation_status"
    ]
    == "passed"
).all():
    raise ValueError(
        "At least one sampled Model C kernel candidate "
        "failed validation."
    )

kernel_family_validation.to_csv(
    KERNEL_FAMILY_VALIDATION_PATH,
    index=False,
)

display(rho_candidate_table)

display(
    kernel_family_validation.groupby(
        "sequence_kernel",
        as_index=False,
    ).agg(
        candidate_rho_values=(
            "rho",
            "count",
        ),
        minimum_sample_eigenvalue=(
            "sample_minimum_eigenvalue",
            "min",
        ),
        validation_status=(
            "validation_status",
            lambda values: (
                "passed"
                if (values == "passed").all()
                else "failed"
            ),
        ),
    )
)

print(f"Saved: {RHO_CANDIDATE_PATH}")
print(f"Saved: {KERNEL_FAMILY_VALIDATION_PATH}")

print(
    "\nNo rho value was selected in this cell. "
    "Selection is deferred to BioSample-grouped "
    "pathogen-out validation."
)

print(
    "\nTransition: Cell 25.8 will save the two sequence "
    "components and the complete kernel-family configuration."
)


In [ ]:
# =============================================================================
# Cell 25.8
# =============================================================================

#@title Cell 25.8 - Save the aligned kernel components and configuration
# This cell saves persistent copies of both sequence kernels beside the Model
# 3B subset and records the exact formula, dimensions, pathogen order, rho grid
# and later validation rule.

BASE_SEQUENCE_COMPONENT_PATH = (
    NOTEBOOK25_RESULT_DIRECTORY
    / "25_sequence_kernel_base_weighted.npy"
)

LOCUS_SEQUENCE_COMPONENT_PATH = (
    NOTEBOOK25_RESULT_DIRECTORY
    / "25_sequence_kernel_locus_weighted.npy"
)

KERNEL_CONFIGURATION_PATH = (
    NOTEBOOK25_RESULT_DIRECTORY
    / "25_model_c_kernel_family_configuration.json"
)


def copy_file_atomically(
    source_path,
    destination_path,
):
    partial_path = destination_path.with_suffix(
        destination_path.suffix + ".partial"
    )

    partial_path.unlink(
        missing_ok=True
    )

    shutil.copy2(
        source_path,
        partial_path,
    )

    if (
        partial_path.stat().st_size
        != Path(source_path).stat().st_size
    ):
        raise IOError(
            f"The copied file has the wrong size: "
            f"{destination_path.name}"
        )

    partial_path.replace(
        destination_path
    )


copy_file_atomically(
    notebook24_input_paths[
        "24_sequence_kernel.npy"
    ],
    BASE_SEQUENCE_COMPONENT_PATH,
)

copy_file_atomically(
    notebook24_input_paths[
        "24_sequence_kernel_locus_weighted.npy"
    ],
    LOCUS_SEQUENCE_COMPONENT_PATH,
)


kernel_family_configuration = {
    "notebook": 25,
    "model": "Model C",
    "model_c_pathogens":
        EXPECTED_MODEL_C_PATHOGENS,
    "kernel_dimensions": [
        EXPECTED_MODEL_C_PATHOGENS,
        EXPECTED_MODEL_C_PATHOGENS,
    ],
    "kernel_dtype": "float32",
    "combination_formula": (
        "K_P_C(rho) = rho * K_seq + "
        "(1 - rho) * K_P_3B"
    ),
    "rho_interval": [
        0.0,
        1.0,
    ],
    "rho_candidates": [
        float(rho)
        for rho in RHO_CANDIDATES
    ],
    "sequence_kernel_candidates": [
        "base-weighted",
        "locus-weighted",
    ],
    "component_files": {
        "model3b_subset":
            MODEL3B_SUBSET_KERNEL_PATH.name,
        "base_weighted_sequence":
            BASE_SEQUENCE_COMPONENT_PATH.name,
        "locus_weighted_sequence":
            LOCUS_SEQUENCE_COMPONENT_PATH.name,
        "pathogen_order":
            MODEL_C_PATHOGEN_ORDER_PATH.name,
    },
    "selection_method": (
        "Joint selection of sequence-kernel type and rho "
        "using BioSample-grouped pathogen-out validation"
    ),
    "rho_selected_in_notebook25": False,
    "mic_outcomes_used_in_notebook25": False,
    "source_archives": {
        "model3b": model3b_archive_path.name,
        "sequence": notebook24_archive_path.name,
    },
}

temporary_configuration_path = (
    KERNEL_CONFIGURATION_PATH.with_suffix(
        ".json.partial"
    )
)

with open(
    temporary_configuration_path,
    "w",
    encoding="utf-8",
) as configuration_file:
    json.dump(
        kernel_family_configuration,
        configuration_file,
        indent=2,
    )

temporary_configuration_path.replace(
    KERNEL_CONFIGURATION_PATH
)


saved_component_summary = pd.DataFrame(
    [
        {
            "component": "Model 3B subset",
            "dimensions": "9,058 × 9,058",
            "file_size_MB": round(
                MODEL3B_SUBSET_KERNEL_PATH.stat().st_size
                / (1024 ** 2),
                3,
            ),
        },
        {
            "component": "Base-weighted sequence kernel",
            "dimensions": "9,058 × 9,058",
            "file_size_MB": round(
                BASE_SEQUENCE_COMPONENT_PATH.stat().st_size
                / (1024 ** 2),
                3,
            ),
        },
        {
            "component": "Locus-weighted sequence kernel",
            "dimensions": "9,058 × 9,058",
            "file_size_MB": round(
                LOCUS_SEQUENCE_COMPONENT_PATH.stat().st_size
                / (1024 ** 2),
                3,
            ),
        },
    ]
)

display(saved_component_summary)

print(f"Saved: {BASE_SEQUENCE_COMPONENT_PATH}")
print(f"Saved: {LOCUS_SEQUENCE_COMPONENT_PATH}")
print(f"Saved: {KERNEL_CONFIGURATION_PATH}")

print(
    "\nTransition: Cell 25.9 will validate the saved "
    "Notebook 25 files and record their checksums."
)


In [ ]:
# =============================================================================
# Cell 25.9
# =============================================================================

#@title Cell 25.9 - Validate the saved Notebook 25 outputs
# This cell reopens the three saved component matrices, confirms their shapes
# and pathogen order, calculates file checksums and records the final output
# validation status before packaging.

saved_component_paths = {
    "Model 3B subset":
        MODEL3B_SUBSET_KERNEL_PATH,
    "Base-weighted sequence kernel":
        BASE_SEQUENCE_COMPONENT_PATH,
    "Locus-weighted sequence kernel":
        LOCUS_SEQUENCE_COMPONENT_PATH,
}

saved_component_validation_rows = []

for component_name, component_path in (
    saved_component_paths.items()
):
    saved_kernel = np.load(
        component_path,
        mmap_mode="r",
    )

    shape_correct = (
        saved_kernel.shape
        == (
            EXPECTED_MODEL_C_PATHOGENS,
            EXPECTED_MODEL_C_PATHOGENS,
        )
    )

    if not shape_correct:
        raise ValueError(
            f"{component_name} has the wrong saved "
            f"dimensions: {saved_kernel.shape}."
        )

    saved_component_validation_rows.append(
        {
            "component": component_name,
            "rows": saved_kernel.shape[0],
            "columns": saved_kernel.shape[1],
            "size_bytes":
                component_path.stat().st_size,
            "sha256":
                file_sha256(component_path),
            "validation_status": "passed",
        }
    )

saved_order = pd.read_csv(
    MODEL_C_PATHOGEN_ORDER_PATH
)

if len(saved_order) != EXPECTED_MODEL_C_PATHOGENS:
    raise ValueError(
        "The saved Model C pathogen order has the wrong "
        "number of rows."
    )

if not np.array_equal(
    saved_order[
        "model_c_row_index"
    ].to_numpy(dtype=np.int64),
    expected_model_c_rows,
):
    raise ValueError(
        "The saved Model C pathogen order is not complete "
        "and consecutive."
    )

with open(
    KERNEL_CONFIGURATION_PATH,
    "r",
    encoding="utf-8",
) as configuration_file:
    saved_configuration = json.load(
        configuration_file
    )

if saved_configuration[
    "rho_selected_in_notebook25"
]:
    raise ValueError(
        "Notebook 25 must not select rho."
    )

SAVED_COMPONENT_VALIDATION_PATH = (
    NOTEBOOK25_RESULT_DIRECTORY
    / "25_saved_component_file_validation.csv"
)

OUTPUT_VALIDATION_PATH = (
    NOTEBOOK25_RESULT_DIRECTORY
    / "25_output_validation.csv"
)

saved_component_file_validation = pd.DataFrame(
    saved_component_validation_rows
)

saved_component_file_validation.to_csv(
    SAVED_COMPONENT_VALIDATION_PATH,
    index=False,
)

output_validation = pd.DataFrame(
    [
        {
            "metric": "Model C pathogens",
            "value": EXPECTED_MODEL_C_PATHOGENS,
        },
        {
            "metric": "Aligned kernel components",
            "value": 3,
        },
        {
            "metric": "Sequence-kernel candidates",
            "value": 2,
        },
        {
            "metric": "Candidate rho values",
            "value": len(RHO_CANDIDATES),
        },
        {
            "metric": "Candidate kernel combinations",
            "value": (
                2 * len(RHO_CANDIDATES)
            ),
        },
        {
            "metric": "BioSample order validated",
            "value": True,
        },
        {
            "metric": "Rho selected",
            "value": False,
        },
        {
            "metric": "Notebook 25 validation status",
            "value": "passed",
        },
    ]
)

output_validation.to_csv(
    OUTPUT_VALIDATION_PATH,
    index=False,
)

display(saved_component_file_validation)
display(output_validation)

print(f"Saved: {SAVED_COMPONENT_VALIDATION_PATH}")
print(f"Saved: {OUTPUT_VALIDATION_PATH}")

print(
    "\nTransition: Cell 25.10 will package the validated "
    "Notebook 25 outputs and report the final status."
)


In [ ]:
# =============================================================================
# Cell 25.10
# =============================================================================

#@title Cell 25.10 - Package and report the final Notebook 25 outputs
# This cell creates one validated ZIP archive containing the three aligned
# kernel components, pathogen order, rho grid, configuration and validation
# tables required for later BioSample-grouped pathogen-out selection.

FINAL_OUTPUT_ARCHIVE_PATH = (
    NOTEBOOK25_DIRECTORY
    / "25_model_c_pathogen_kernel_candidate_components.zip"
)

OUTPUT_MANIFEST_PATH = (
    NOTEBOOK25_RESULT_DIRECTORY
    / "25_output_manifest.json"
)

files_to_package = [
    MODEL3B_SUBSET_KERNEL_PATH,
    BASE_SEQUENCE_COMPONENT_PATH,
    LOCUS_SEQUENCE_COMPONENT_PATH,
    MODEL_C_PATHOGEN_ORDER_PATH,
    RHO_CANDIDATE_PATH,
    KERNEL_COMPONENT_VALIDATION_PATH,
    KERNEL_FAMILY_VALIDATION_PATH,
    KERNEL_CONFIGURATION_PATH,
    SAVED_COMPONENT_VALIDATION_PATH,
    OUTPUT_VALIDATION_PATH,
]

missing_output_files = [
    path
    for path in files_to_package
    if not path.exists()
]

if missing_output_files:
    raise FileNotFoundError(
        "Notebook 25 output files are missing: "
        f"{missing_output_files}"
    )

output_manifest = {
    "notebook": 25,
    "model": "Model C",
    "model_c_pathogens":
        EXPECTED_MODEL_C_PATHOGENS,
    "kernel_dimensions": [
        EXPECTED_MODEL_C_PATHOGENS,
        EXPECTED_MODEL_C_PATHOGENS,
    ],
    "component_kernels": [
        MODEL3B_SUBSET_KERNEL_PATH.name,
        BASE_SEQUENCE_COMPONENT_PATH.name,
        LOCUS_SEQUENCE_COMPONENT_PATH.name,
    ],
    "combination_formula": (
        "K_P_C(rho) = rho * K_seq + "
        "(1 - rho) * K_P_3B"
    ),
    "rho_candidates": [
        float(rho)
        for rho in RHO_CANDIDATES
    ],
    "sequence_kernel_candidates": [
        "base-weighted",
        "locus-weighted",
    ],
    "selection_status": (
        "Deferred to BioSample-grouped pathogen-out "
        "validation"
    ),
    "source_archives": [
        {
            "file_name": model3b_archive_path.name,
            "sha256": file_sha256(
                model3b_archive_path
            ),
        },
        {
            "file_name": notebook24_archive_path.name,
            "sha256": file_sha256(
                notebook24_archive_path
            ),
        },
    ],
    "files": [
        {
            "file_name": path.name,
            "size_bytes": path.stat().st_size,
            "sha256": file_sha256(path),
        }
        for path in files_to_package
    ],
}

temporary_manifest_path = (
    OUTPUT_MANIFEST_PATH.with_suffix(
        ".json.partial"
    )
)

with open(
    temporary_manifest_path,
    "w",
    encoding="utf-8",
) as manifest_file:
    json.dump(
        output_manifest,
        manifest_file,
        indent=2,
    )

temporary_manifest_path.replace(
    OUTPUT_MANIFEST_PATH
)

files_to_package.append(
    OUTPUT_MANIFEST_PATH
)

local_archive_path = (
    WORK_DIRECTORY
    / FINAL_OUTPUT_ARCHIVE_PATH.name
)

local_archive_path.unlink(
    missing_ok=True
)

with zipfile.ZipFile(
    local_archive_path,
    "w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=1,
    allowZip64=True,
) as archive:
    for file_path in files_to_package:
        archive.write(
            file_path,
            arcname=file_path.name,
        )

with zipfile.ZipFile(
    local_archive_path,
    "r",
) as archive:
    damaged_member = archive.testzip()

    if damaged_member is not None:
        raise ValueError(
            "The final archive contains a damaged file: "
            f"{damaged_member}"
        )

    archived_members = set(
        archive.namelist()
    )

expected_members = {
    path.name
    for path in files_to_package
}

if archived_members != expected_members:
    raise ValueError(
        "The final archive member list is incomplete."
    )

local_archive_sha256 = file_sha256(
    local_archive_path
)

partial_archive_path = (
    FINAL_OUTPUT_ARCHIVE_PATH.with_suffix(
        ".zip.partial"
    )
)

partial_archive_path.unlink(
    missing_ok=True
)

shutil.copy2(
    local_archive_path,
    partial_archive_path,
)

if (
    file_sha256(partial_archive_path)
    != local_archive_sha256
):
    raise IOError(
        "The copied final archive does not match "
        "the locally validated archive."
    )

partial_archive_path.replace(
    FINAL_OUTPUT_ARCHIVE_PATH
)

with zipfile.ZipFile(
    FINAL_OUTPUT_ARCHIVE_PATH,
    "r",
) as archive:
    if archive.testzip() is not None:
        raise ValueError(
            "The saved final archive failed validation."
        )


final_summary = pd.DataFrame(
    [
        {
            "metric": "Model C pathogens",
            "value": EXPECTED_MODEL_C_PATHOGENS,
        },
        {
            "metric": "Aligned kernel components",
            "value": 3,
        },
        {
            "metric": "Sequence-kernel candidates",
            "value": 2,
        },
        {
            "metric": "Candidate rho values",
            "value": len(RHO_CANDIDATES),
        },
        {
            "metric": "Candidate kernel combinations",
            "value": (
                2 * len(RHO_CANDIDATES)
            ),
        },
        {
            "metric": "Rho selection status",
            "value": (
                "Deferred to BioSample-grouped "
                "pathogen-out validation"
            ),
        },
        {
            "metric": "Final archive size (GB)",
            "value": round(
                FINAL_OUTPUT_ARCHIVE_PATH.stat().st_size
                / (1024 ** 3),
                3,
            ),
        },
        {
            "metric": "Notebook 25 validation status",
            "value": "passed",
        },
    ]
)

display(final_summary)

print(f"Saved: {FINAL_OUTPUT_ARCHIVE_PATH}")

print(
    "\nNotebook 25 completed successfully. "
    "The aligned Model 3B and sequence-kernel components "
    "are ready for BioSample-grouped pathogen-out selection."
)
